In [20]:
import cv2
import numpy as np
import math

point4 = []
src_img = cv2.imread('card.jpg')
COLOR = (0, 0, 255)

def mouse_handler(event, x, y, flag, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        point4.append((x,y))
        if len(point4) == 4:
            show_result()
    
        for point in point4:
            cv2.circle(src_img, point , 4 , COLOR, cv2.FILLED )
    
        cv2.imshow('img', src_img)
    if event == cv2.EVENT_MOUSEMOVE:
        if 0<len(point4)<4:
            # 원본 이미지를 복사하여 임시 이미지 생성 (잔상 제거용)
            img_copy = src_img.copy()
            cv2.line(img_copy, point4[-1], (x, y), (0, 255, 0), 2, cv2.LINE_AA)
            cv2.imshow('img', img_copy)
        
  
def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def show_result():

    dists = []
    for i in range(4):
        d = distance(point4[i], point4[(i+1) % 4])
        dists.append(d)
    
    # 가로, 세로 구하기
    height = int(max(dists))
    width = int(min(dists))

    src = np.float32(point4)
    dst = np.array([[0,0],[width, 0],[width, height],[0, height]], dtype=np.float32)
    # 4개 꼭지점은 시계방향으로 좌표값 입력
    
    matrix = cv2.getPerspectiveTransform(src, dst)
    res = cv2.warpPerspective(src_img, matrix , (width, height))
    cv2.imshow('result', res)

cv2.namedWindow('img')
cv2.setMouseCallback('img', mouse_handler)

cv2.imshow('img', src_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np
import math

# 변수 초기화
point4 = []
# 원본 이미지는 건드리지 않고 보존합니다.
src_img = cv2.imread('card.jpg') 
COLOR = (0, 0, 255)

def mouse_handler(event, x, y, flag, param):
    global src_img
    img_copy = src_img.copy()
    # 1. 마우스 클릭 이벤트
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(point4) < 4:
            point4.append((x, y))
            # 클릭한 지점에 점 그리기 (원본에 반영)
            cv2.circle(src_img, (x, y), 5, COLOR, cv2.FILLED)
            
            # 이전 점과 현재 점 연결 (원본에 반영)
            if len(point4) > 1:
                cv2.line(src_img, point4[-2], point4[-1], COLOR, 2)
                
            cv2.imshow('img', src_img)

        if len(point4) == 4:
            show_result()

    # 2. 마우스 이동 이벤트 (가이드 라인)
    if event == cv2.EVENT_MOUSEMOVE:
        if 0 < len(point4) < 4:
            # 원본 이미지를 복사해서 가이드 라인을 그림 (잔상 방지)
            # img_copy = src_img.copy()
            # 마지막 클릭 지점과 현재 마우스 좌표(x, y) 연결
            cv2.line(img_copy, point4[-1], (x, y), COLOR, 2, cv2.LINE_AA)
            cv2.imshow('img', img_copy)

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def show_result():
    # 시계 방향 순서대로 좌표가 찍혔다고 가정 (좌상, 우상, 우하, 좌하)
    src = np.float32(point4)
    
    # 가로 세로 길이를 단순 거리 기반으로 계산
    width = int(max(distance(src[0], src[1]), distance(src[2], src[3])))
    height = int(max(distance(src[0], src[3]), distance(src[1], src[2])))

    dst = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)
    
    matrix = cv2.getPerspectiveTransform(src, dst)
    res = cv2.warpPerspective(src_img, matrix, (width, height))
    cv2.imshow('result', res)

cv2.namedWindow('img')
cv2.setMouseCallback('img', mouse_handler)

cv2.imshow('img', src_img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [25]:
import cv2
import numpy as np
import math

# 변수 초기화
point4 = []
# 원본 이미지는 건드리지 않고 보존합니다.
origin_img = cv2.imread('card.jpg') 
COLOR = (0, 255, 0)

def mouse_handler(event, x, y, flag, param):
    if event == cv2.EVENT_LBUTTONDOWN:
        if len(point4) < 4:
            point4.append((x, y))
        
        if len(point4) == 4:
            show_result()

    # 화면 갱신 로직 (마우스가 움직이거나 클릭할 때마다)
    if len(point4) < 4:
        # 매번 원본에서 복사본을 만들어 '그리기용'으로만 씁니다.
        display_img = origin_img.copy()
        
        # 1. 이미 클릭한 점들 그리기
        for p in point4:
            cv2.circle(display_img, p, 5, COLOR, cv2.FILLED)
        
        # 2. 이미 클릭한 점들 사이의 선 연결
        for i in range(len(point4) - 1):
            cv2.line(display_img, point4[i], point4[i+1], COLOR, 2)
            
        # 3. 마우스 이동 시 가이드 라인 (마지막 점 ~ 현재 마우스 위치)
        if event == cv2.EVENT_MOUSEMOVE and len(point4) > 0:
            cv2.line(display_img, point4[-1], (x, y), COLOR, 1, cv2.LINE_AA)
            
        cv2.imshow('img', display_img)

def distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def show_result():
    src = np.float32(point4)
    
    # 가로, 세로 계산 (이전 대화의 정석 로직 적용)
    width = int(max(distance(src[0], src[1]), distance(src[2], src[3])))
    height = int(max(distance(src[0], src[3]), distance(src[1], src[2])))

    dst = np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)
    
    matrix = cv2.getPerspectiveTransform(src, dst)
    
    # 핵심!! 점이 찍히지 않은 'origin_img'를 사용하여 변환합니다.
    res = cv2.warpPerspective(origin_img, matrix, (width, height))
    cv2.imshow('result', res)

cv2.namedWindow('img')
cv2.setMouseCallback('img', mouse_handler)

# 초기 화면 출력
cv2.imshow('img', origin_img)
cv2.waitKey(0)
cv2.destroyAllWindows()